# Final Project - Faster R-CNN Road Object Detection

This notebook expands the earlier R-CNN assignment into a six-class BDD100K detector. The CPU profiles use Faster R-CNN MobileNetV3-FPN; the final accuracy profile uses Faster R-CNN ResNet50-FPN V2.

The new prediction head is initialized from matching COCO classes (car, bus, truck, person, traffic light, and stop sign as a traffic-sign starting point). This retains more useful transfer learning than randomly replacing the complete head. Evaluation reports true 101-point AP, mAP@50, and mAP@50:95.

In [ ]:
from pathlib import Path
from dataclasses import asdict
import json, random, sys, time

import cv2
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Subset
from tqdm.auto import tqdm

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from road_detection.constants import PROJECT_CLASSES
from road_detection.rcnn_dataset import YoloDetectionDataset, collate_fn
from road_detection.rcnn_metrics import collect_predictions, evaluate_predictions, tune_score_threshold
from road_detection.rcnn_model import build_faster_rcnn

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Project: {PROJECT_ROOT}')
print(f'PyTorch: {torch.__version__} | device: {DEVICE}')

## Run profile

`cpu_quick` is the default end-to-end CPU experiment. `cpu_practical` trains on a larger subset. `accuracy` is the full ResNet50 comparison and should normally run on a GPU.

In [ ]:
RUN_MODE = 'cpu_quick'  # cpu_quick | cpu_practical | accuracy
TARGET_F1 = 0.80

PROFILES = {
    'cpu_quick': dict(variant='mobilenet', min_size=480, max_size=640, batch=1,
                      epochs=6, lr=3e-4, trainable_backbone_layers=2,
                      max_train=600, max_val=150, max_test=150, workers=0),
    'cpu_practical': dict(variant='mobilenet', min_size=576, max_size=768, batch=2,
                          epochs=12, lr=2e-4, trainable_backbone_layers=3,
                          max_train=4000, max_val=800, max_test=800, workers=0),
    'accuracy': dict(variant='resnet50', min_size=800, max_size=1024, batch=2,
                     epochs=24, lr=2e-4, trainable_backbone_layers=5,
                     max_train=None, max_val=None, max_test=None, workers=6),
}
cfg = PROFILES[RUN_MODE]
DATASET_ROOT = PROJECT_ROOT / 'data' / 'bdd100k_yolo'
OUTPUT_DIR = PROJECT_ROOT / 'runs' / 'notebooks' / 'rcnn' / RUN_MODE
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
BEST_WEIGHTS = OUTPUT_DIR / 'best_fasterrcnn.pth'
cfg

## Data loaders

Training receives color jitter and horizontal flips. Validation and test are deterministic. Random subsets are seeded so the CPU profiles remain reproducible and are not biased toward the first driving sequence.

In [ ]:
def seeded_subset(dataset, limit, seed):
    if limit is None or limit >= len(dataset):
        return dataset
    generator = torch.Generator().manual_seed(seed)
    indices = torch.randperm(len(dataset), generator=generator)[:limit].tolist()
    return Subset(dataset, indices)

train_full = YoloDetectionDataset(DATASET_ROOT, 'train', max_size=cfg['max_size'], augment=True)
val_full = YoloDetectionDataset(DATASET_ROOT, 'val', max_size=cfg['max_size'], augment=False)
test_full = YoloDetectionDataset(DATASET_ROOT, 'test', max_size=cfg['max_size'], augment=False)
train_ds = seeded_subset(train_full, cfg['max_train'], SEED)
val_ds = seeded_subset(val_full, cfg['max_val'], SEED + 1)
test_ds = seeded_subset(test_full, cfg['max_test'], SEED + 2)

train_loader = DataLoader(train_ds, batch_size=cfg['batch'], shuffle=True,
                          num_workers=cfg['workers'], collate_fn=collate_fn,
                          pin_memory=DEVICE.type == 'cuda')
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False,
                        num_workers=cfg['workers'], collate_fn=collate_fn)
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False,
                         num_workers=cfg['workers'], collate_fn=collate_fn)
pd.DataFrame({'split': ['train', 'validation', 'test'],
              'images': [len(train_ds), len(val_ds), len(test_ds)]})

In [ ]:
image, target = train_ds[0]
fig, axis = plt.subplots(figsize=(12, 7))
axis.imshow(image.permute(1, 2, 0))
for box, label in zip(target['boxes'], target['labels']):
    x1, y1, x2, y2 = box.tolist()
    axis.add_patch(Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, color='#ff3b30', linewidth=2))
    axis.text(x1, y1, PROJECT_CLASSES[int(label) - 1], color='white', backgroundcolor='#ff3b30')
axis.set_title('Training sample after augmentation')
axis.axis('off')
plt.show()

## Model and optimizer

In [ ]:
model = build_faster_rcnn(
    num_classes=len(PROJECT_CLASSES) + 1,
    variant=cfg['variant'], min_size=cfg['min_size'], max_size=cfg['max_size'],
    trainable_backbone_layers=cfg['trainable_backbone_layers'],
    pretrained=True, class_names=PROJECT_CLASSES, transfer_coco_head=True,
).to(DEVICE)
trainable_parameters = [parameter for parameter in model.parameters() if parameter.requires_grad]
optimizer = torch.optim.AdamW(trainable_parameters, lr=cfg['lr'], weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg['epochs'])
scaler = torch.amp.GradScaler('cuda', enabled=DEVICE.type == 'cuda')
print(f'Trainable parameters: {sum(p.numel() for p in trainable_parameters):,}')

## Train and validate

The best checkpoint is selected by validation F1. mAP is calculated with class-aware IoU matching and 101-point interpolation.

In [ ]:
history = []
best_f1 = -1.0
for epoch in range(1, cfg['epochs'] + 1):
    model.train()
    running_loss = 0.0
    progress = tqdm(train_loader, desc=f'Epoch {epoch}/{cfg["epochs"]}')
    for images, targets in progress:
        images = [image.to(DEVICE) for image in images]
        targets = [{key: value.to(DEVICE) for key, value in target.items()} for target in targets]
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type=DEVICE.type, enabled=DEVICE.type == 'cuda'):
            losses = model(images, targets)
            loss = sum(losses.values())
        scaler.scale(loss).backward()
        torch.nn.utils.clip_grad_norm_(trainable_parameters, 5.0)
        scaler.step(optimizer)
        scaler.update()
        running_loss += float(loss.detach().cpu())
        progress.set_postfix(loss=f'{running_loss / max(1, progress.n):.4f}')
    scheduler.step()

    val_predictions, val_targets = collect_predictions(model, val_loader, DEVICE)
    val_metrics = evaluate_predictions(val_predictions, val_targets, score_threshold=0.25,
                                       class_names=PROJECT_CLASSES)
    metric_values = asdict(val_metrics)
    metric_values.pop('per_class')
    row = {'epoch': epoch, 'train_loss': running_loss / max(1, len(train_loader)), **metric_values}
    history.append(row)
    display(pd.DataFrame([row]).round(4))

    if val_metrics.f1 > best_f1:
        best_f1 = val_metrics.f1
        torch.save({
            'model': model.state_dict(), 'classes': PROJECT_CLASSES, 'epoch': epoch,
            'metrics': row, 'variant': cfg['variant'], 'min_size': cfg['min_size'],
            'max_size': cfg['max_size'], 'run_mode': RUN_MODE,
        }, BEST_WEIGHTS)
        print(f'Saved best checkpoint: {BEST_WEIGHTS}')

In [ ]:
history_df = pd.DataFrame(history)
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
history_df.plot(x='epoch', y='train_loss', marker='o', ax=axes[0], title='Training loss')
history_df.plot(x='epoch', y=['precision', 'recall', 'f1'], marker='o', ax=axes[1], title='Validation P/R/F1')
history_df.plot(x='epoch', y=['map50', 'map50_95'], marker='o', ax=axes[2], title='Validation mAP')
for axis in axes: axis.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## Tune on validation, evaluate once on test

The confidence threshold is chosen only on validation data and then held fixed for test data. This avoids tuning on the test set.

In [ ]:
checkpoint = torch.load(BEST_WEIGHTS, map_location=DEVICE, weights_only=False)
model.load_state_dict(checkpoint['model'])
model.eval()

val_predictions, val_targets = collect_predictions(model, val_loader, DEVICE)
tuned_val = tune_score_threshold(val_predictions, val_targets, class_names=PROJECT_CLASSES)
test_predictions, test_targets = collect_predictions(model, test_loader, DEVICE)
test_metrics = evaluate_predictions(test_predictions, test_targets,
                                    score_threshold=tuned_val.score_threshold,
                                    class_names=PROJECT_CLASSES)

summary = pd.DataFrame([
    {'split': 'validation', **{key: value for key, value in asdict(tuned_val).items() if key != 'per_class'}},
    {'split': 'test', **{key: value for key, value in asdict(test_metrics).items() if key != 'per_class'}},
])
display(summary.style.format({column: '{:.3f}' for column in ['precision', 'recall', 'f1', 'map50', 'map50_95', 'score_threshold']}))

In [ ]:
def class_table(metrics):
    return pd.DataFrame([{'class': name, **values} for name, values in metrics.per_class.items()])

print('Validation by class')
display(class_table(tuned_val).style.format({c: '{:.3f}' for c in ['precision', 'recall', 'f1', 'ap50', 'map50_95']}))
print('Test by class')
display(class_table(test_metrics).style.format({c: '{:.3f}' for c in ['precision', 'recall', 'f1', 'ap50', 'map50_95']}))

In [ ]:
acceptance = summary[['split', 'f1']].copy()
acceptance['target'] = TARGET_F1
acceptance['passed'] = acceptance['f1'] >= TARGET_F1
display(acceptance)
if not acceptance['passed'].all():
    print('The measured 0.80 F1 target has not yet been met. Increase the profile, inspect weak classes, and retrain.')
else:
    print('The 0.80 validation/test F1 target is met.')

## Prediction examples and low-light failure review

In [ ]:
COLORS = ['#2878b5', '#2ca02c', '#d62728', '#9467bd', '#ff7f0e', '#17becf']

def show_predictions(indices, title):
    images = [test_ds[index][0] for index in indices]
    with torch.inference_mode():
        outputs = model([image.to(DEVICE) for image in images])
    fig, axes = plt.subplots(2, 3, figsize=(16, 9))
    for axis, image, output in zip(axes.flat, images, outputs):
        axis.imshow(image.permute(1, 2, 0))
        for box, label, score in zip(output['boxes'].cpu(), output['labels'].cpu(), output['scores'].cpu()):
            if float(score) < tuned_val.score_threshold: continue
            x1, y1, x2, y2 = box.tolist(); class_id = int(label) - 1
            axis.add_patch(Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False,
                                     color=COLORS[class_id], linewidth=2))
            axis.text(x1, y1, f'{PROJECT_CLASSES[class_id]} {float(score):.2f}',
                      color='white', backgroundcolor=COLORS[class_id])
        axis.axis('off')
    for axis in axes.flat[len(images):]: axis.axis('off')
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

sample_indices = random.Random(SEED).sample(range(len(test_ds)), min(6, len(test_ds)))
show_predictions(sample_indices, 'Held-out test predictions')

In [ ]:
brightness_rows = []
for index in range(len(test_ds)):
    image, _ = test_ds[index]
    brightness_rows.append((float(image.mean()), index))
dark_indices = [index for _, index in sorted(brightness_rows)[:min(6, len(brightness_rows))]]
show_predictions(dark_indices, 'Low-light failure-case review')

## Inference speed and final artifacts

In [ ]:
benchmark_count = min(25, len(test_ds))
benchmark_images = [test_ds[index][0].to(DEVICE) for index in range(benchmark_count)]
started = time.perf_counter()
with torch.inference_mode():
    for image in benchmark_images:
        _ = model([image])
elapsed = time.perf_counter() - started
fps = benchmark_count / max(elapsed, 1e-9)
print(f'{benchmark_count} images in {elapsed:.2f}s = {fps:.2f} FPS on {DEVICE}')

report = {
    'run_mode': RUN_MODE, 'weights': str(BEST_WEIGHTS), 'target_f1': TARGET_F1,
    'inference_fps': fps, 'validation': asdict(tuned_val), 'test': asdict(test_metrics),
}
checkpoint['score_threshold'] = tuned_val.score_threshold
checkpoint['final_evaluation'] = report
torch.save(checkpoint, BEST_WEIGHTS)
history_df.to_csv(OUTPUT_DIR / 'training_history.csv', index=False)
(OUTPUT_DIR / 'final_evaluation.json').write_text(json.dumps(report, indent=2), encoding='utf-8')
print(f'Saved checkpoint and report under {OUTPUT_DIR}')